In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:latest")
print('LLM is ready')

LLM is ready


In [3]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="Write a poem on Paris",
)
response = model.invoke(prompt_template.format())
print(response)

content="Amidst the Seine's gentle flow,\nA city of love, where hearts glow,\nParis, the City of Light,\nWhere art and beauty take flight.\n\nThe Eiffel Tower's iron lace,\nA symbol of romance, a wondrous place,\nWhere lovers stroll, hand in hand,\nUnderneath the city's enchanted land.\n\nThe Louvre's halls, a treasure trove,\nWhere masterpieces, the world can love,\nFrom Mona Lisa's smile, to Venus' might,\nA cultural legacy, shining bright.\n\nMontmartre's streets, a bohemian dream,\nWhere artists' souls, forever beam,\nThe scent of croissants, wafts through the air,\nAs café culture, beckons without a care.\n\nThe Seine's riverbanks, a peaceful stroll,\nWhere the city's charm, forever unfolds,\nThe sound of jazz, or French cabaret,\nA symphony, that echoes through the night's dark heart.\n\nFrom Notre-Dame's spires, to the Arc de Triomphe's grandeur,\nParis, a city, of elegance and power,\nA place where dreams, and hopes, entwine,\nWhere the essence of France, forever will shine.\n\

In [4]:
from langchain_core.output_parsers import StrOutputParser
str_parser = StrOutputParser()
chain = prompt_template | model | str_parser
response = chain.invoke({})
print(response)

Amidst the Seine's gentle flow,
Lies a city of love and glow,
Paris, the city of dreams so fair,
Where art and beauty linger in the air.

The Eiffel Tower stands tall and high,
A symbol of love that touches the sky,
Its iron latticework a work of art,
A masterpiece that beats within the heart.

The Louvre's halls, a treasure trove,
Of Mona Lisa's smile, a mystery to prove,
The Venus de Milo, a beauty to behold,
A testament to ancient tales, forever to be told.

Montmartre's streets, a winding delight,
A haven for artists, in the morning light,
The Sacré-Cœur, a church so white,
A beacon of hope, in the city's delight.

The Seine's banks, a promenade so fine,
A place to stroll, and let the heart entwine,
With the beauty of the city, and all its charm,
A place to fall in love, and let the heart disarm.

The cafes, a haven for the soul,
A place to sip, and let the heart unfold,
The croissants, a treat so divine,
A taste of heaven, that's simply sublime.

Paris, the city of love, so true,


In [ ]:

from langchain_core.output_parsers import ListOutputParser

class SemicolonSeparatedList(ListOutputParser):
    def parse(self, text: str) -> list[str]:
        return [item.strip() for item in text.split(';') if item.strip()]

prompt_template = PromptTemplate(
    template="Give me list of semicolon separated 5 names of important cities of India"
)
list_parser = SemicolonSeparatedList()
chain = prompt_template | model | list_parser
response = chain.invoke({})
print(response)

['Mumbai', 'New Delhi', 'Chennai', 'Bengaluru', 'Hyderabad']


In [13]:

from langchain_core.output_parsers import ListOutputParser

class SemicolonSeparatedList(ListOutputParser):
    def parse(self, text: str) -> list[str]:
        return [item.strip() for item in text.split(';') if item.strip()]

list_parser = SemicolonSeparatedList()
prompt_template = PromptTemplate(
    template="Give me list of semicolon separated 5 names of important cities of India"
)
structured_model = model.with_structured_output(SemicolonSeparatedList)
response = structured_model.invoke("Give me list of semicolon separated 5 names of important cities of India")
print(response)

In [15]:
# Json output parser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Account(BaseModel):
    accountno: int = Field(description="Account No"),
    name: str = Field(description="Account Name"),
    balance: float = Field("Account Balance")

json_parser = JsonOutputParser(pydantic_object=Account)

prompt_template = PromptTemplate(
    template="Extract the account details of a person having Account no {accno} belongs to Tom and having balance 10000" \
    "\nformat instructions={format_instructions}",
    input_variables=["accno"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)
prompt = prompt_template.format(accno=234)
response = model.invoke(prompt)
print(type(response))
print(response)

account_json = json_parser.parse(response.content)
print(f'account JSON: {account_json}')

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Account No'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Account Name'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


<class 'langchain_core.messages.ai.AIMessage'>
content='{"accountno": 234, "name": "Tom", "balance": 10000}' additional_kwargs={} response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-08-31T09:34:36.039278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2752647600, 'load_duration': 610482200, 'prompt_eval_count': 347, 'prompt_eval_duration': 109836000, 'eval_count': 21, 'eval_duration': 2023815000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'} id='lc_run--01a0572b-a543-78b2-b0fe-75a6bb7c6a9e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 347, 'output_tokens': 21, 'total_tokens': 368}
account JSON: {'accountno': 234, 'name': 'Tom', 'balance': 10000}


In [4]:
# Pydantic output parser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Account(BaseModel):
    accountno: int = Field(description="Account No"),
    name: str = Field(description="Account Name"),
    balance: float = Field("Account Balance")

pydantic_parser = PydanticOutputParser(pydantic_object=Account)

prompt_template = PromptTemplate(
    template="Extract the account details of a person having Account no {accno} belongs to Tom and having balance 10000" \
    "\nformat instructions={format_instructions}",
    input_variables=["accno"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)
prompt = prompt_template.format(accno=234)
response = model.invoke(prompt)
print(type(response))
print(response)

account = pydantic_parser.parse(response.content)
print(f'account: {account}')

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Account No'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Account Name'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


<class 'langchain_core.messages.ai.AIMessage'>
content='Here\'s the JSON formatted account details:\n\n```\n{\n  "properties": {\n    "accountno": {\n      "title": "Accountno",\n      "type": "integer"\n    },\n    "name": {\n      "title": "Name",\n      "type": "string"\n    },\n    "balance": {\n      "default": "Account Balance",\n      "title": "Balance",\n      "type": "number"\n    }\n  },\n  "required": ["accountno", "name", "balance"],\n  "accountno": 234,\n  "name": "Tom",\n  "balance": 10000\n}\n```\n\nThis follows the provided schema with `accountno`, `name`, and `balance` included in the `required` list and the actual values assigned to the `accountno`, `name`, and `balance` properties.' additional_kwargs={} response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-09-01T08:43:20.5950195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22758198800, 'load_duration': 455799700, 'prompt_eval_count': 226, 'prompt_eval_duration': 5384162000, 'eval_count': 176